# 第二章：Cypher 查询语言

## 学习目标

- 掌握 Cypher 的 CREATE / MATCH / MERGE / DELETE 语法
- 理解模式匹配（Pattern Matching）思维
- 使用 WHERE 过滤和 RETURN 投影
- 掌握关系遍历和变长路径查询
- 对比 SQL 与 Cypher 的思维差异

## 0. 环境准备

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USERNAME"], os.environ["NEO4J_PASSWORD"]),
)
driver.verify_connectivity()
print("✓ Neo4j 连接成功")

In [ ]:
# 清空数据库，确保本章从零开始
driver.execute_query("MATCH (n) DETACH DELETE n")
print("✓ 数据库已清空")

## 1. 准备电影数据集

本章会用一个小型电影知识图谱来演示所有 Cypher 操作。数据包括：
- **Movie** 节点：电影（标题、年份、类型）
- **Person** 节点：演员和导演（姓名、出生年份）
- **ACTED_IN** 关系：演员出演电影（带角色属性）
- **DIRECTED** 关系：导演执导电影
- **KNOWS** 关系：人与人之间的社交关系

In [ ]:
# 创建电影知识图谱
driver.execute_query("""
    // 创建电影节点
    CREATE (matrix:Movie {title: '黑客帝国', year: 1999, genre: '科幻'})
    CREATE (inception:Movie {title: '盗梦空间', year: 2010, genre: '科幻'})
    CREATE (forrest:Movie {title: '阿甘正传', year: 1994, genre: '剧情'})
    CREATE (django:Movie {title: '被解救的姜戈', year: 2012, genre: '西部'})

    // 创建演员和导演
    CREATE (keanu:Person {name: '基努·里维斯', born: 1964})
    CREATE (leo:Person {name: '莱昂纳多', born: 1974})
    CREATE (tom:Person {name: '汤姆·汉克斯', born: 1956})
    CREATE (nolan:Person {name: '克里斯托弗·诺兰', born: 1970})
    CREATE (tarantino:Person {name: '昆汀·塔伦蒂诺', born: 1963})
    CREATE (wachowski:Person {name: '沃卓斯基', born: 1965})

    // 创建 ACTED_IN 关系（带角色属性）
    CREATE (keanu)-[:ACTED_IN {role: 'Neo'}]->(matrix)
    CREATE (leo)-[:ACTED_IN {role: 'Cobb'}]->(inception)
    CREATE (leo)-[:ACTED_IN {role: 'Django'}]->(django)
    CREATE (tom)-[:ACTED_IN {role: 'Forrest'}]->(forrest)

    // 创建 DIRECTED 关系
    CREATE (nolan)-[:DIRECTED]->(inception)
    CREATE (tarantino)-[:DIRECTED]->(django)
    CREATE (wachowski)-[:DIRECTED]->(matrix)

    // 创建社交关系
    CREATE (keanu)-[:KNOWS]->(leo)
    CREATE (leo)-[:KNOWS]->(tom)
""")
print("✓ 电影知识图谱创建完成")

### 刚才发生了什么？

一条 `CREATE` 语句构建了整张图。注意几点：
- 变量名（`matrix`、`keanu` 等）只在**当前语句**内有效，用来在同一条语句内引用节点
- 关系上也可以带属性：`[:ACTED_IN {role: 'Neo'}]`
- `//` 是 Cypher 的注释语法

In [ ]:
def run_query(query, params=None):
    """执行 Cypher 查询并以表格形式打印结果"""
    records, summary, keys = driver.execute_query(query, parameters_=params)
    if not records:
        print("(无结果)")
        return records
    # 打印表头
    print(" | ".join(keys))
    print("-" * (len(" | ".join(keys)) + 10))
    for record in records:
        print(" | ".join(str(record[k]) for k in keys))
    return records

`run_query` 是一个辅助函数：执行 Cypher、把结果打印成简易表格。后面所有查询都会用到它。

## 2. MATCH：模式匹配查询

Cypher 最核心的能力就是**模式匹配**。你用 ASCII 画出想要的图结构，Neo4j 帮你找到所有匹配的数据。

基本语法：
```cypher
MATCH (变量:标签 {属性过滤})
RETURN 变量.属性
```

In [ ]:
# 查询所有电影
print("所有电影：")
run_query("MATCH (m:Movie) RETURN m.title AS 电影, m.year AS 年份, m.genre AS 类型")

In [ ]:
# 查询所有演员和他们出演的电影
print("演员和电影：")
run_query("""
    MATCH (p:Person)-[:ACTED_IN]->(m:Movie)
    RETURN p.name AS 演员, m.title AS 电影
""")

In [ ]:
# 查询每部电影的导演
print("电影的导演：")
run_query("""
    MATCH (d:Person)-[:DIRECTED]->(m:Movie)
    RETURN m.title AS 电影, d.name AS 导演
""")

### 刚才发生了什么？

MATCH 语句用 **ASCII-art 模式**来描述你要查找的图结构：

| 符号 | 含义 | 示例 |
|------|------|------|
| `(n)` | 节点 | `(m:Movie)` |
| `(n:Label)` | 带标签的节点 | `(p:Person)` |
| `-->` | 有向关系 | `(a)-->(b)` |
| `-[:TYPE]->` | 带类型的关系 | `-[:ACTED_IN]->` |
| `<--` | 反向关系 | `(a)<--(b)` |

**核心思想**：你不需要告诉数据库"去哪张表找、怎么 JOIN"，而是**画出你想要的模式**，让数据库去匹配。

## 3. WHERE / ORDER BY / LIMIT

和 SQL 一样，Cypher 也用 `WHERE` 过滤结果，但支持更多图特有的操作。

In [ ]:
# WHERE 条件过滤 + 排序
print("2000年之后的电影：")
run_query("""
    MATCH (m:Movie)
    WHERE m.year > 2000
    RETURN m.title AS 电影, m.year AS 年份
    ORDER BY m.year DESC
""")

In [ ]:
# 字符串匹配：CONTAINS / STARTS WITH / ENDS WITH
print("名字包含'纳'的人：")
run_query("""
    MATCH (p:Person)
    WHERE p.name CONTAINS '纳'
    RETURN p.name AS 姓名
""")

In [ ]:
# 多条件组合
print("科幻电影（2000年后）：")
run_query("""
    MATCH (m:Movie)
    WHERE m.genre = '科幻' AND m.year > 2000
    RETURN m.title AS 电影, m.year AS 年份
""")

### 刚才发生了什么？

| 操作 | Cypher 语法 | 说明 |
|------|------------|------|
| 比较 | `>`, `<`, `=`, `<>` | 和 SQL 一样 |
| 字符串包含 | `CONTAINS '子串'` | 比 SQL 的 `LIKE '%子串%'` 更直观 |
| 字符串开头 | `STARTS WITH '前缀'` | |
| 字符串结尾 | `ENDS WITH '后缀'` | |
| 逻辑组合 | `AND`, `OR`, `NOT` | |
| 排序 | `ORDER BY ... DESC/ASC` | 默认 ASC |
| 限制条数 | `LIMIT n` | |

## 4. CREATE：创建数据

上面用一条大 CREATE 批量创建了数据。实际开发中，更常见的是**先查已有节点，再创建关系**。

In [ ]:
# 创建新电影节点
driver.execute_query("""
    CREATE (m:Movie {title: '星际穿越', year: 2014, genre: '科幻'})
    RETURN m.title
""")
print("✓ 已创建新电影：星际穿越")

In [ ]:
# 连接已有节点：让诺兰执导星际穿越
driver.execute_query("""
    MATCH (d:Person {name: '克里斯托弗·诺兰'})
    MATCH (m:Movie {title: '星际穿越'})
    CREATE (d)-[:DIRECTED]->(m)
""")
print("✓ 已创建导演关系")

In [ ]:
# 验证结果
print("诺兰执导的电影：")
run_query("""
    MATCH (d:Person {name: '克里斯托弗·诺兰'})-[:DIRECTED]->(m:Movie)
    RETURN m.title AS 电影, m.year AS 年份
""")

### 刚才发生了什么？

创建关系的典型模式是 **MATCH + CREATE**：
1. `MATCH` 找到已存在的节点
2. `CREATE` 在它们之间建立关系

这比在一条语句里同时创建所有东西更灵活。但 CREATE 有一个问题——如果重复执行会怎样？

## 5. MERGE vs CREATE（幂等性）

`CREATE` 每次执行都会创建新数据，不管是否已经存在。这在脚本重跑、程序重试时会产生重复数据。

`MERGE` 解决了这个问题：**有则匹配，无则创建**。

In [ ]:
# ❌ CREATE 每次都会创建新节点（产生重复）
driver.execute_query("CREATE (p:Person {name: '测试用户'})")
driver.execute_query("CREATE (p:Person {name: '测试用户'})")

records, _, _ = driver.execute_query("""
    MATCH (p:Person {name: '测试用户'}) RETURN count(p) AS 数量
""")
print(f"❌ CREATE 两次后: {records[0]['数量']} 个节点（产生了重复！）")

In [ ]:
# 清理刚才的重复数据
driver.execute_query("MATCH (p:Person {name: '测试用户'}) DETACH DELETE p")

# ✅ MERGE 有则匹配，无则创建（幂等）
driver.execute_query("MERGE (p:Person {name: '测试用户'})")
driver.execute_query("MERGE (p:Person {name: '测试用户'})")

records, _, _ = driver.execute_query("""
    MATCH (p:Person {name: '测试用户'}) RETURN count(p) AS 数量
""")
print(f"✅ MERGE 两次后: {records[0]['数量']} 个节点（没有重复）")

# 清理测试数据
driver.execute_query("MATCH (p:Person {name: '测试用户'}) DETACH DELETE p")

### 刚才发生了什么？

| | CREATE | MERGE |
|---|---|---|
| **行为** | 无条件创建新数据 | 先查找，找到就复用，找不到才创建 |
| **重复执行** | 产生重复节点 | 不会重复（幂等） |
| **适用场景** | 确定是新数据 | 不确定是否已存在 |
| **类比** | SQL `INSERT` | SQL `INSERT ... ON CONFLICT DO NOTHING` |

**经验法则**：在生产代码中，优先使用 `MERGE` 而非 `CREATE`，除非你明确知道数据一定是新的。

### 常见问题

**Q: MERGE 匹配的是哪些属性？**

MERGE 会匹配你写在模式里的**所有属性**。例如 `MERGE (p:Person {name: '张三', age: 30})` 要求 name 和 age 都一致才算"找到"。如果只想按 name 匹配，用：
```cypher
MERGE (p:Person {name: '张三'})
ON CREATE SET p.age = 30
```

`ON CREATE SET` 只在新建时设置属性，已存在时不更新。对应的 `ON MATCH SET` 则只在匹配到时更新。

## 6. 更新数据（SET / REMOVE）

In [ ]:
# SET 添加/更新属性
driver.execute_query("""
    MATCH (m:Movie {title: '黑客帝国'})
    SET m.rating = 8.7
""")
print("✓ 已添加评分")

# 查看结果
run_query("MATCH (m:Movie {title: '黑客帝国'}) RETURN m.title AS 电影, m.rating AS 评分")

In [ ]:
# REMOVE 删除属性
driver.execute_query("""
    MATCH (m:Movie {title: '黑客帝国'})
    REMOVE m.rating
""")
print("✓ 已移除评分")

# 验证：rating 已不存在
run_query("MATCH (m:Movie {title: '黑客帝国'}) RETURN m.title AS 电影, m.rating AS 评分")

### 刚才发生了什么？

| 操作 | 语法 | 说明 |
|------|------|------|
| 添加/更新属性 | `SET n.prop = value` | 属性不存在就添加，存在就覆盖 |
| 删除属性 | `REMOVE n.prop` | 属性被彻底移除（不是设为 null） |
| 添加标签 | `SET n:NewLabel` | 一个节点可以有多个标签 |
| 删除标签 | `REMOVE n:OldLabel` | |

注意：图数据库的 Schema 是灵活的，同一个标签下的节点不需要有相同的属性。`rating` 只有黑客帝国有，其他电影没有，完全没问题。

## 7. 删除数据（DELETE / DETACH DELETE）

Neo4j 不允许删除还有关系连着的节点——这是一个安全机制，防止产生"悬挂关系"。

In [ ]:
# 创建临时测试数据
driver.execute_query("""
    CREATE (t:TempNode {name: '临时节点'})
    CREATE (t2:TempNode {name: '临时节点2'})
    CREATE (t)-[:TEMP_REL]->(t2)
""")
print("✓ 已创建临时数据（两个节点 + 一条关系）")

In [ ]:
# ❌ DELETE 不能删除有关系的节点
try:
    driver.execute_query("MATCH (t:TempNode) DELETE t")
except Exception as e:
    print(f"❌ DELETE 失败: {type(e).__name__}")
    print(f"   原因: 有关系的节点不能直接 DELETE")

In [ ]:
# ✅ DETACH DELETE 先断开所有关系，再删除节点
driver.execute_query("MATCH (t:TempNode) DETACH DELETE t")
print("✅ DETACH DELETE 成功")

# 验证
records, _, _ = driver.execute_query("MATCH (t:TempNode) RETURN count(t) AS 数量")
print(f"   剩余 TempNode: {records[0]['数量']} 个")

### 刚才发生了什么？

| 操作 | 说明 |
|------|------|
| `DELETE n` | 删除节点/关系，但节点必须没有关系 |
| `DETACH DELETE n` | 先删除节点的所有关系，再删除节点本身 |

**安全建议**：清库时用 `MATCH (n) DETACH DELETE n`，日常操作时小心使用 `DETACH DELETE`，最好加上标签或 WHERE 条件限制范围。

## 8. 路径查询

图数据库最强大的能力之一是**路径遍历**。还记得第一章"朋友的朋友"的例子吗？Cypher 用变长路径 `[*min..max]` 来表达。

In [ ]:
# 变长路径：从基努出发，沿 KNOWS 关系走 1 到 2 跳
print("从基努·里维斯出发的 KNOWS 路径（1-2 跳）：")
run_query("""
    MATCH (start:Person {name: '基努·里维斯'})-[:KNOWS*1..2]->(end:Person)
    RETURN start.name AS 起点, end.name AS 终点
""")

### 刚才发生了什么？

`[:KNOWS*1..2]` 表示沿 `KNOWS` 关系走 **1 到 2 步**：
- 1 步：基努 → 莱昂纳多
- 2 步：基努 → 莱昂纳多 → 汤姆·汉克斯

语法解析：
- `*1..2` — 最少 1 步，最多 2 步
- `*..3` — 最少 1 步，最多 3 步
- `*2` — 恰好 2 步
- `*` — 任意步数（慎用！大图上可能很慢）

In [ ]:
# 最短路径：基努·里维斯 和 汤姆·汉克斯 之间的最短路径
print("基努·里维斯 → 汤姆·汉克斯 的最短路径：")
run_query("""
    MATCH path = shortestPath(
        (a:Person {name: '基努·里维斯'})-[*]-(b:Person {name: '汤姆·汉克斯'})
    )
    RETURN [n IN nodes(path) |
        CASE WHEN n:Person THEN n.name WHEN n:Movie THEN n.title END
    ] AS 路径
""")

### 刚才发生了什么？

`shortestPath()` 找两个节点之间的最短路径。注意这里用的是 `-[*]-`（无方向、任意关系类型），因为我们不限制中间经过什么类型的关系。

`nodes(path)` 提取路径上的所有节点。用 `CASE WHEN` 根据标签显示不同的属性（Person 显示 name，Movie 显示 title）。

### 常见问题

**Q: 变长路径会不会很慢？**

会的。`[*]`（无限制步数）在大图上可能遍历整张图。实践中：
- 总是设置上限：`[*..5]` 而不是 `[*]`
- 限制关系类型：`[:KNOWS*1..3]` 而不是 `[*1..3]`
- 大图上考虑用 APOC 的路径算法

## 9. 对比：SQL 思维 vs Cypher 思维

如果你有 SQL 背景，下面的对照表帮助你快速建立映射：

| SQL | Cypher | 说明 |
|-----|--------|------|
| `SELECT` | `RETURN` | 指定返回字段 |
| `FROM table` | `MATCH (pattern)` | 指定数据来源 |
| `WHERE` | `WHERE` | 过滤条件 |
| `JOIN` | `-[rel]->` | 关联数据 |
| `INSERT INTO` | `CREATE` | 新增数据 |
| `UPDATE SET` | `SET` | 更新数据 |
| `DELETE FROM` | `DELETE / DETACH DELETE` | 删除数据 |
| `INSERT ... ON CONFLICT` | `MERGE` | 幂等写入 |

### 思维方式的根本差异

**SQL 思维**：数据住在表里，查询就是"从哪张表取、怎么 JOIN、过滤什么条件"。

```sql
-- 查找莱昂纳多演过的电影
SELECT m.title
FROM persons p
JOIN acted_in a ON p.id = a.person_id
JOIN movies m ON a.movie_id = m.id
WHERE p.name = '莱昂纳多';
```

**Cypher 思维**：数据就是图，查询就是"画出你想要的模式"。

```cypher
-- 同样的查询
MATCH (p:Person {name: '莱昂纳多'})-[:ACTED_IN]->(m:Movie)
RETURN m.title
```

关键区别：
- SQL 需要你知道表结构和外键关系，手动写 JOIN
- Cypher 直接用模式描述数据关系，**你画的就是你要的**
- 关联层数越深，差异越大（3 层 JOIN vs `[*3]`）

## 10. Cypher 速查表

| 操作 | 语法 | 示例 |
|------|------|------|
| 创建节点 | `CREATE (n:Label {prop: val})` | `CREATE (p:Person {name: '张三'})` |
| 创建关系 | `CREATE (a)-[:TYPE]->(b)` | `(a)-[:KNOWS]->(b)` |
| 查询 | `MATCH (pattern) RETURN ...` | `MATCH (p:Person) RETURN p.name` |
| 过滤 | `WHERE condition` | `WHERE p.age > 30` |
| 排序 | `ORDER BY ... DESC/ASC` | `ORDER BY m.year DESC` |
| 限制 | `LIMIT n` | `LIMIT 10` |
| 幂等创建 | `MERGE (pattern)` | `MERGE (p:Person {name: '张三'})` |
| 更新属性 | `SET n.prop = val` | `SET p.age = 31` |
| 删除属性 | `REMOVE n.prop` | `REMOVE p.age` |
| 删除节点 | `DETACH DELETE n` | `DETACH DELETE p` |
| 变长路径 | `[*min..max]` | `[:KNOWS*1..3]` |
| 最短路径 | `shortestPath((a)-[*]-(b))` | 见第 8 节示例 |

In [ ]:
# 本章数据保留给下一章使用（不清理）
driver.close()
print("✓ 连接已关闭")

## 总结

本章学习了 Cypher 查询语言的核心操作：

- ✅ **CREATE** 创建节点和关系
- ✅ **MATCH** 模式匹配查询——Cypher 的核心能力
- ✅ **WHERE / ORDER BY / LIMIT** 过滤、排序、分页
- ✅ **MERGE vs CREATE** 幂等性对比
- ✅ **SET / REMOVE** 更新和删除属性
- ✅ **DELETE / DETACH DELETE** 删除节点和关系
- ✅ **变长路径查询**和**最短路径**
- ✅ **SQL 与 Cypher** 的思维对比

最重要的思维转变：**Cypher 是声明式的模式描述语言，你画出想要的模式，数据库帮你找到匹配的数据。**

## 下一章预告

**第三章：数据建模与批量导入** —— 手动 CREATE 适合小数据量。真实项目中需要从 Python 数据（字典、JSON）批量导入，并设计合理的图数据模型。下一章学习 UNWIND 批量写入、事务管理和约束索引。